# Small Language Model Recursive Latent Reasoner

An end-to-end, self-contained research notebook for training a small causal Transformer to solve algorithmic composition tasks using either direct decoding or an adaptive recurrent latent-reasoning loop. The implementation logs reproducible artifacts, keeps train/validation/test generators disjoint, evaluates depth generalization, and compares direct, fixed-depth, adaptive-depth, and no-recurrence controls.

## Scope and claims

This notebook tests a narrow, measurable claim: whether a learned small language model benefits from recurrent latent computation on held-out function-composition problems. It does not establish general reasoning, real-world task competence, or superiority over other architectures.

Run top-to-bottom in a fresh Jupyter or Colab runtime. GPU is recommended but not required.

In [ ]:
# Environment, reproducibility, and immutable run configuration
import json, math, os, platform, random, sys, time, hashlib, zipfile
from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 20260914
RUN_ID = datetime.now(timezone.utc).strftime('slm_rlr_%Y%m%dT%H%M%SZ')
OUT = Path('slm_rlr_runs') / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

@dataclass(frozen=True)
class Config:
    vocab_symbols: int = 16
    max_hops: int = 8
    train_hops: tuple = (1, 2, 3, 4)
    validation_hops: tuple = (1, 2, 3, 4)
    test_hops: tuple = (1, 2, 3, 4, 5, 6, 7, 8)
    train_examples: int = 24000
    validation_examples: int = 3000
    test_examples_per_hop: int = 1000
    batch_size: int = 128
    d_model: int = 192
    n_heads: int = 6
    n_layers: int = 4
    mlp_ratio: int = 4
    dropout: float = 0.0
    latent_steps: int = 6
    min_latent_steps: int = 1
    max_latent_steps: int = 8
    halt_threshold: float = 0.80
    epochs: int = 30
    learning_rate: float = 3e-4
    weight_decay: float = 0.05
    warmup_fraction: float = 0.08
    grad_clip: float = 1.0
    seed: int = SEED

CFG = Config()
(OUT / 'config.json').write_text(json.dumps(asdict(CFG), indent=2, default=list))
print({'run_id': RUN_ID, 'output_dir': str(OUT), 'device': str(DEVICE), 'torch': torch.__version__})
if torch.cuda.is_available():
    print({'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda})

{'run_id': 'slm_rlr_20260914T110123Z', 'output_dir': 'slm_rlr_runs/slm_rlr_20260914T110123Z', 'device': 'cuda', 'torch': '2.11.0+cu128'}
{'gpu': 'NVIDIA A100-SXM4-40GB', 'cuda': '12.8'}


## Task, oracle, and tokenizer

An input is a sequence of ordered lookup tables (random permutations) and a start symbol. The required answer is the result of applying the tables in order. The training model sees a tokenized textual-style representation only; the NumPy oracle remains separate from the model.

The held-out test set includes chain lengths beyond training depth, so the depth-generalization result should be read separately from in-distribution validation accuracy.

In [ ]:
PAD, BOS, EOS, SEP, START, ANSWER, TABLE = range(7)
SYMBOL_OFFSET = 7
VOCAB_SIZE = SYMBOL_OFFSET + CFG.vocab_symbols
TOKEN_NAMES = {PAD: 'PAD', BOS: 'BOS', EOS: 'EOS', SEP: 'SEP', START: 'START', ANSWER: 'ANSWER', TABLE: 'TABLE'}

def symbol_token(x):
    return SYMBOL_OFFSET + int(x)

def token_symbol(tok):
    return int(tok) - SYMBOL_OFFSET

def compose_oracle(permutations, start):
    value = int(start)
    trace = [value]
    for permutation in permutations:
        value = int(permutation[value])
        trace.append(value)
    return value, trace

def make_example(rng, hops):
    perms = [rng.permutation(CFG.vocab_symbols).astype(np.int64) for _ in range(hops)]
    start = int(rng.integers(CFG.vocab_symbols))
    answer, trace = compose_oracle(perms, start)
    return {'hops': int(hops), 'start': start, 'target': answer, 'trace': trace, 'permutations': [p.tolist() for p in perms]}

def encode_example(example):
    tokens = [BOS, START, symbol_token(example['start']), SEP]
    for permutation in example['permutations']:
        tokens.extend([TABLE] + [symbol_token(v) for v in permutation] + [SEP])
    tokens.extend([ANSWER, symbol_token(example['target']), EOS])
    answer_position = len(tokens) - 2
    return tokens, answer_position

MAX_SEQ_LEN = len(encode_example(make_example(np.random.default_rng(0), CFG.max_hops))[0])
print({'vocab_size': VOCAB_SIZE, 'max_sequence_length': MAX_SEQ_LEN})

class CompositionDataset(Dataset):
    def __init__(self, n, hops, seed):
        rng = np.random.default_rng(seed)
        choices = np.asarray(hops, dtype=np.int64)
        self.examples = [make_example(rng, int(rng.choice(choices))) for _ in range(n)]
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        tokens, answer_position = encode_example(ex)
        return {'tokens': torch.tensor(tokens, dtype=torch.long), 'answer_position': answer_position, 'target': ex['target'], 'hops': ex['hops'], 'raw': ex}

def collate(batch):
    width = max(item['tokens'].numel() for item in batch)
    x = torch.full((len(batch), width), PAD, dtype=torch.long)
    mask = torch.zeros((len(batch), width), dtype=torch.bool)
    for i, item in enumerate(batch):
        n = item['tokens'].numel()
        x[i, :n] = item['tokens']
        mask[i, :n] = True
    return {
        'tokens': x,
        'attention_mask': mask,
        'answer_position': torch.tensor([item['answer_position'] for item in batch], dtype=torch.long),
        'target': torch.tensor([item['target'] for item in batch], dtype=torch.long),
        'hops': torch.tensor([item['hops'] for item in batch], dtype=torch.long),
        'raw': [item['raw'] for item in batch],
    }

train_ds = CompositionDataset(CFG.train_examples, CFG.train_hops, seed=101)
valid_ds = CompositionDataset(CFG.validation_examples, CFG.validation_hops, seed=202)
test_sets = {h: CompositionDataset(CFG.test_examples_per_hop, (h,), seed=10_000 + h) for h in CFG.test_hops}
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, collate_fn=collate, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=CFG.batch_size, shuffle=False, collate_fn=collate, num_workers=0)
test_loaders = {h: DataLoader(ds, batch_size=CFG.batch_size, shuffle=False, collate_fn=collate, num_workers=0) for h, ds in test_sets.items()}
print({'train': len(train_ds), 'validation': len(valid_ds), 'test_total': sum(len(x) for x in test_sets.values())})

{'vocab_size': 23, 'max_sequence_length': 151}
{'train': 24000, 'validation': 3000, 'test_total': 8000}


## Small language model with recurrent latent reasoning

The encoder is a compact causal Transformer. Its answer-position state becomes the latent state. A shared gated transition block is then applied repeatedly, with a learned halting head. At inference, the same trained model can be evaluated with a fixed number of updates, adaptive halting, or zero updates. This is a practical RLR-style latent loop, not a proof that hidden states are interpretable symbolic traces.

In [ ]:
class CausalBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_model * mlp_ratio), nn.GELU(), nn.Linear(d_model * mlp_ratio, d_model), nn.Dropout(dropout))
    def forward(self, x, padding_mask):
        t = x.size(1)
        causal = torch.triu(torch.ones(t, t, device=x.device, dtype=torch.bool), diagonal=1)
        h = self.norm1(x)
        a, _ = self.attn(h, h, h, attn_mask=causal, key_padding_mask=~padding_mask, need_weights=False)
        x = x + a
        return x + self.mlp(self.norm2(x))

class LatentTransition(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.gate = nn.Linear(d_model, d_model)
        self.update = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))
    def forward(self, z):
        h = self.norm(z)
        gate = torch.sigmoid(self.gate(h))
        candidate = self.update(h)
        return gate * z + (1.0 - gate) * candidate

class SLMRLR(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.token_embed = nn.Embedding(VOCAB_SIZE, cfg.d_model)
        self.pos_embed = nn.Embedding(MAX_SEQ_LEN, cfg.d_model)
        self.blocks = nn.ModuleList([CausalBlock(cfg.d_model, cfg.n_heads, cfg.mlp_ratio, cfg.dropout) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.transition = LatentTransition(cfg.d_model)
        self.answer_head = nn.Linear(cfg.d_model, CFG.vocab_symbols)
        self.halt_head = nn.Sequential(nn.LayerNorm(cfg.d_model), nn.Linear(cfg.d_model, 1))
    def encode(self, tokens, attention_mask, answer_position):
        positions = torch.arange(tokens.size(1), device=tokens.device).unsqueeze(0)
        x = self.token_embed(tokens) + self.pos_embed(positions)
        for block in self.blocks:
            x = block(x, attention_mask)
        x = self.final_norm(x)
        return x[torch.arange(x.size(0), device=x.device), answer_position]
    def forward(self, tokens, attention_mask, answer_position, latent_steps=None, adaptive=False, halt_threshold=None):
        z = self.encode(tokens, attention_mask, answer_position)
        steps = self.cfg.latent_steps if latent_steps is None else int(latent_steps)
        threshold = self.cfg.halt_threshold if halt_threshold is None else float(halt_threshold)
        active = torch.ones(z.size(0), dtype=torch.bool, device=z.device)
        used = torch.zeros(z.size(0), dtype=torch.long, device=z.device)
        halt_probs = []
        states = [z]
        for step in range(steps):
            proposal = self.transition(z)
            z = torch.where(active[:, None], proposal, z) if adaptive else proposal
            p_halt = torch.sigmoid(self.halt_head(z)).squeeze(-1)
            halt_probs.append(p_halt)
            used = used + active.long() if adaptive else used + 1
            if adaptive and step + 1 >= self.cfg.min_latent_steps:
                active = active & (p_halt < threshold)
            states.append(z)
        return {'logits': self.answer_head(z), 'halt_probs': torch.stack(halt_probs, 1) if halt_probs else z.new_zeros((z.size(0), 0)), 'steps_used': used, 'states': states}

model = SLMRLR(CFG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print({'parameters': n_params, 'parameters_millions': round(n_params / 1e6, 3)})

{'parameters': 2150225, 'parameters_millions': 2.15}


## Training protocol

Supervision is applied to the final answer. A small auxiliary loss teaches the halting head to prefer at least as many latent steps as the chain length, clipped to the training loop budget. This is deliberately disclosed: adaptive halting is not learned from answer loss alone in this baseline.

In [ ]:
def to_device(batch):
    return {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}

def loss_fn(output, target, hops):
    answer_loss = F.cross_entropy(output['logits'], target)
    hp = output['halt_probs']
    if hp.size(1) == 0:
        return answer_loss, {'answer_loss': float(answer_loss.detach()), 'halt_loss': 0.0}
    desired = torch.clamp(hops, min=1, max=hp.size(1))
    step_ids = torch.arange(1, hp.size(1) + 1, device=hp.device).unsqueeze(0)
    halt_targets = (step_ids >= desired.unsqueeze(1)).float()
    halt_loss = F.binary_cross_entropy(hp, halt_targets)
    loss = answer_loss + 0.10 * halt_loss
    return loss, {'answer_loss': float(answer_loss.detach()), 'halt_loss': float(halt_loss.detach())}

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay, betas=(0.9, 0.95))
total_updates = CFG.epochs * len(train_loader)
warmup_updates = max(1, int(CFG.warmup_fraction * total_updates))
def lr_scale(step):
    if step < warmup_updates:
        return (step + 1) / warmup_updates
    progress = (step - warmup_updates) / max(1, total_updates - warmup_updates)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_scale)

@torch.no_grad()
def evaluate(loader, latent_steps, adaptive=False):
    model.eval()
    totals = defaultdict(int)
    records = []
    for batch in loader:
        batch = to_device(batch)
        out = model(batch['tokens'], batch['attention_mask'], batch['answer_position'], latent_steps=latent_steps, adaptive=adaptive)
        pred = out['logits'].argmax(-1)
        for i in range(pred.numel()):
            h = int(batch['hops'][i])
            ok = bool(pred[i].item() == batch['target'][i].item())
            totals[(h, 'n')] += 1
            totals[(h, 'correct')] += int(ok)
            records.append({
                'hops': h,
                'start': batch['raw'][i]['start'],
                'target': int(batch['target'][i]),
                'prediction': int(pred[i]),
                'correct': ok,
                'steps_used': int(out['steps_used'][i]),
                'oracle_trace': batch['raw'][i]['trace'],
                'permutations': batch['raw'][i]['permutations'],
            })
    summary = {}
    for h in sorted({k[0] for k in totals if k[1] == 'n'}):
        n, correct = totals[(h, 'n')], totals[(h, 'correct')]
        summary[h] = {'n': n, 'correct': correct, 'accuracy': correct / n}
    return summary, records

history = []
best_state = None
best_valid = -1.0
global_step = 0
for epoch in range(1, CFG.epochs + 1):
    model.train()
    running_loss, running_correct, running_n = 0.0, 0, 0
    for batch in train_loader:
        batch = to_device(batch)
        optimizer.zero_grad(set_to_none=True)
        out = model(batch['tokens'], batch['attention_mask'], batch['answer_position'], latent_steps=CFG.latent_steps, adaptive=False)
        loss, components = loss_fn(out, batch['target'], batch['hops'])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        optimizer.step()
        scheduler.step()
        running_loss += float(loss.detach()) * batch['target'].numel()
        running_correct += int((out['logits'].argmax(-1) == batch['target']).sum())
        running_n += batch['target'].numel()
        global_step += 1
    val_summary, _ = evaluate(valid_loader, latent_steps=CFG.latent_steps, adaptive=False)
    val_correct = sum(row['correct'] for row in val_summary.values())
    val_n = sum(row['n'] for row in val_summary.values())
    val_acc = val_correct / val_n
    row = {'epoch': epoch, 'train_loss': running_loss / running_n, 'train_accuracy': running_correct / running_n, 'validation_accuracy': val_acc, 'lr': optimizer.param_groups[0]['lr']}
    history.append(row)
    print(row)
    if val_acc > best_valid:
        best_valid = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

assert best_state is not None
model.load_state_dict(best_state)
torch.save({'config': asdict(CFG), 'state_dict': best_state, 'best_validation_accuracy': best_valid}, OUT / 'best_model.pt')
(OUT / 'training_history.json').write_text(json.dumps(history, indent=2))
print({'best_validation_accuracy': best_valid, 'checkpoint': str(OUT / 'best_model.pt')})

{'epoch': 1, 'train_loss': 1.2917165926198164, 'train_accuracy': 0.8345416666666666, 'validation_accuracy': 1.0, 'lr': 0.00012572062084257205}
{'epoch': 2, 'train_loss': 0.0038107489833491856, 'train_accuracy': 1.0, 'validation_accuracy': 1.0, 'lr': 0.0002507760532150776}
{'epoch': 3, 'train_loss': 0.01346404454090225, 'train_accuracy': 0.99725, 'validation_accuracy': 1.0, 'lr': 0.00029968419160870247}
{'epoch': 4, 'train_loss': 4.656864999409057e-06, 'train_accuracy': 1.0, 'validation_accuracy': 1.0, 'lr': 0.0002977645398373083}
{'epoch': 5, 'train_loss': 8.105620768598479e-08, 'train_accuracy': 1.0, 'validation_accuracy': 1.0, 'lr': 0.0002941267372692767}
{'epoch': 6, 'train_loss': 0.0002832433960094439, 'train_accuracy': 1.0, 'validation_accuracy': 1.0, 'lr': 0.0002888178619375517}
{'epoch': 7, 'train_loss': 0.00016230784263213612, 'train_accuracy': 1.0, 'validation_accuracy': 1.0, 'lr': 0.00028190661779268843}
{'epoch': 8, 'train_loss': 8.220049879052264e-08, 'train_accuracy': 1.0,

## Controls and held-out evaluation

All conditions use the same learned encoder and answer head. The comparison isolates how many latent transition applications are permitted: no recurrent update, one update, fixed training-budget updates, and adaptive updates. This is an intervention on inference computation, not a clean comparison with separately trained architectures.

In [ ]:
conditions = {
    'no_latent_loop': {'latent_steps': 0, 'adaptive': False},
    'one_latent_step': {'latent_steps': 1, 'adaptive': False},
    'fixed_latent_loop': {'latent_steps': CFG.latent_steps, 'adaptive': False},
    'adaptive_latent_loop': {'latent_steps': CFG.max_latent_steps, 'adaptive': True},
}

all_records = []
all_summary = {}
for name, spec in conditions.items():
    per_hop = {}
    for hops, loader in test_loaders.items():
        summary, records = evaluate(loader, **spec)
        row = summary[hops]
        steps = float(np.mean([r['steps_used'] for r in records]))
        row['mean_steps_used'] = steps
        row['condition'] = name
        per_hop[hops] = row
        for record in records:
            record['condition'] = name
        all_records.extend(records)
    all_summary[name] = per_hop

with (OUT / 'heldout_records.jsonl').open('w') as f:
    for r in all_records:
        f.write(json.dumps(r, separators=(',', ':')) + '\n')
(OUT / 'heldout_summary.json').write_text(json.dumps(all_summary, indent=2, sort_keys=True))

print('| Condition | Hops 1-4 | Hops 5-8 | All | Mean adaptive steps |')
print('|---|---:|---:|---:|---:|')
for name, per_hop in all_summary.items():
    in_rows = [per_hop[h] for h in CFG.train_hops]
    out_rows = [per_hop[h] for h in CFG.test_hops if h not in CFG.train_hops]
    all_rows = list(per_hop.values())
    in_acc = sum(r['correct'] for r in in_rows) / sum(r['n'] for r in in_rows)
    out_acc = sum(r['correct'] for r in out_rows) / sum(r['n'] for r in out_rows)
    all_acc = sum(r['correct'] for r in all_rows) / sum(r['n'] for r in all_rows)
    mean_steps = sum(r['mean_steps_used'] * r['n'] for r in all_rows) / sum(r['n'] for r in all_rows)
    print(f'| {name} | {in_acc:.4f} | {out_acc:.4f} | {all_acc:.4f} | {mean_steps:.2f} |')

| Condition | Hops 1-4 | Hops 5-8 | All | Mean adaptive steps |
|---|---:|---:|---:|---:|
| no_latent_loop | 0.5950 | 0.3658 | 0.4804 | 0.00 |
| one_latent_step | 0.9543 | 0.8395 | 0.8969 | 1.00 |
| fixed_latent_loop | 1.0000 | 1.0000 | 1.0000 | 6.00 |
| adaptive_latent_loop | 1.0000 | 1.0000 | 1.0000 | 3.10 |


## Integrity checks, report, and archive

The notebook verifies that every stored target agrees with the independent oracle. It then writes a manifest with hashes. The report is intentionally conservative: it records the evidence generated by this particular run without extrapolating to general language reasoning.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

oracle_failures = []
for record in all_records:
    target, trace = compose_oracle([np.asarray(p, dtype=np.int64) for p in record['permutations']], record['start'])
    if target != record['target'] or trace != record['oracle_trace']:
        oracle_failures.append(record)
if oracle_failures:
    raise RuntimeError(f'FAIL: {len(oracle_failures)} records disagree with the independent oracle')

report = [
    '# SLM Recursive Latent Reasoner Report',
    '',
    f'- Run ID: `{RUN_ID}`',
    f'- Device: `{DEVICE}`',
    f'- Train depths: `{list(CFG.train_hops)}`',
    f'- Held-out test depths: `{list(CFG.test_hops)}`',
    f'- Model parameters: `{n_params:,}`',
    f'- Best validation accuracy: `{best_valid:.4f}`',
    '',
    '## Controls',
    '',
    '| Condition | In-distribution accuracy | Extrapolation accuracy | Overall accuracy |',
    '|---|---:|---:|---:|',
]
for name, per_hop in all_summary.items():
    in_rows = [per_hop[h] for h in CFG.train_hops]
    out_rows = [per_hop[h] for h in CFG.test_hops if h not in CFG.train_hops]
    rows = list(per_hop.values())
    in_acc = sum(r['correct'] for r in in_rows) / sum(r['n'] for r in in_rows)
    out_acc = sum(r['correct'] for r in out_rows) / sum(r['n'] for r in out_rows)
    total_acc = sum(r['correct'] for r in rows) / sum(r['n'] for r in rows)
    report.append(f'| {name} | {in_acc:.4f} | {out_acc:.4f} | {total_acc:.4f} |')
report += [
    '',
    '## Integrity',
    '',
    f'- Independent oracle checks: PASS ({len(all_records):,} records verified).',
    '- Raw held-out predictions and oracle traces are retained in `heldout_records.jsonl`.',
    '- The result demonstrates performance on this generated composition benchmark only.',
]
(OUT / 'REPORT.md').write_text('\n'.join(report) + '\n')

manifest = {
    'run_id': RUN_ID,
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'device': str(DEVICE),
    'config': asdict(CFG),
    'integrity': {'oracle_records_checked': len(all_records), 'oracle_failures': len(oracle_failures), 'status': 'PASS'},
    'files_before_manifest': {p.name: sha256_file(p) for p in OUT.iterdir() if p.is_file()},
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=list))
archive = OUT.parent / f'{RUN_ID}_artifacts.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file():
            z.write(p, arcname=f'{RUN_ID}/{p.name}')

print((OUT / 'REPORT.md').read_text())
print({'manifest': str(OUT / 'manifest.json'), 'archive': str(archive), 'archive_sha256': sha256_file(archive)})

# SLM Recursive Latent Reasoner Report

- Run ID: `slm_rlr_20260914T110123Z`
- Device: `cuda`
- Train depths: `[1, 2, 3, 4]`
- Held-out test depths: `[1, 2, 3, 4, 5, 6, 7, 8]`
- Model parameters: `2,150,225`
- Best validation accuracy: `1.0000`

## Controls

| Condition | In-distribution accuracy | Extrapolation accuracy | Overall accuracy |
|---|---:|---:|---:|
| no_latent_loop | 0.5950 | 0.3658 | 0.4804 |
| one_latent_step | 0.9543 | 0.8395 | 0.8969 |
| fixed_latent_loop | 1.0000 | 1.0000 | 1.0000 |
| adaptive_latent_loop | 1.0000 | 1.0000 | 1.0000 |

## Integrity

- Independent oracle checks: PASS (32,000 records verified).
- Raw held-out predictions and oracle traces are retained in `heldout_records.jsonl`.
- The result demonstrates performance on this generated composition benchmark only.

{'manifest': 'slm_rlr_runs/slm_rlr_20260914T110123Z/manifest.json', 'archive': 'slm_rlr_runs/slm_rlr_20260914T110123Z_artifacts.zip', 'archive_sha256': 'b58ee5a65cec7da47806a4c13a652650a8c25e7dc

In [ ]:
import os
os.kill(os.getpid(), 9)

## Recommended extensions

- Train at least five independent seeds and aggregate mean, standard deviation, and confidence intervals by depth.
- Add compute- and parameter-matched recurrent and non-recurrent baselines trained from scratch, rather than relying only on inference-time ablations.
- Replace the synthetic permutation representation with a task whose latent operations must be inferred from demonstrations, while preserving an independently executable oracle.
- Log training curves, wall-clock cost, token counts, and forward-pass FLOP estimates before making an efficiency claim.
- For interpretability claims, probe whether each recurrent update predicts the corresponding oracle intermediate state using a held-out linear probe.